In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Playground Series S6E7 Kaggle Notebook Cells

## 1. Import + Config

In [ ]:
import os
import gc
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def load_competition_config():
    defaults = {
        "competition": "playground-series-s6e7",
        "metric": "balanced_accuracy",
        "target": "health_condition",
        "end_date": "",
    }
    for path in [Path("competition.json"), Path("../competition.json"), Path("/kaggle/working/competition.json")]:
        if path.exists():
            with open(path) as f:
                config = json.load(f)
            return {**defaults, **config}
    return defaults


COMP_CONFIG = load_competition_config()


class CFG:
    competition = COMP_CONFIG["competition"]
    metric = COMP_CONFIG["metric"]
    target = COMP_CONFIG["target"]
    exp_id = "exp_001"
    parent_exp_id = None
    note = "s6e7 balanced accuracy baseline from ref_exp_028; raw features only"

    data_dir_candidates = [
        Path(f"/kaggle/input/{competition}"),
        Path(f"/kaggle/input/competitions/{competition}"),
        Path("/kaggle/input/playground-series-s6e7"),
        Path("/kaggle/input/competitions/playground-series-s6e7"),
    ]
    submission_path = Path("submission.csv")
    artifact_dir = Path("submissions")

    submitted = False
    public_lb = None
    upload_submission_to_github = True
    save_experiment_to_db = True

    seed = 42
    n_splits = 5
    use_gpu = True
    models = ["lgbm", "mlp"]
    model_weights = None  # None means simple average for non-stacked fallback.
    use_oof_features = False  # Keep old stage2-LGBM path disabled; use LogisticRegression stack below.
    use_logreg_stack = True
    oof_feature_prefix = "stage1"

    num_boost_round = 5000
    early_stopping_rounds = 100
    verbose_eval = 100

    lgbm_params = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "learning_rate": 0.03,
        "num_leaves": 64,
        "max_depth": -1,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "feature_fraction": 0.85,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "min_child_samples": 30,
        "seed": seed,
        "n_jobs": -1,
        "verbosity": -1,
        "device": "gpu" if use_gpu else "cpu",
    }

    xgb_params = {
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",
        "eta": 0.03,
        "max_depth": 7,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "lambda": 1.0,
        "alpha": 0.1,
        "seed": seed,
        "tree_method": "hist",
        "device": "cuda" if use_gpu else "cpu",
    }

    cat_params = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": num_boost_round,
        "learning_rate": 0.03,
        "depth": 7,
        "l2_leaf_reg": 3.0,
        "random_seed": seed,
        "task_type": "GPU" if use_gpu else "CPU",
        "verbose": verbose_eval,
        "allow_writing_files": False,
    }

    mlp_params = {
        "hidden_layer_sizes": (128, 64),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "batch_size": 4096,
        "learning_rate_init": 1e-3,
        "max_iter": 40,
        "early_stopping": True,
        "validation_fraction": 0.1,
        "n_iter_no_change": 5,
        "random_state": seed,
        "verbose": False,
    }

    logreg_stack_params = {
        "C": 1.0,
        "max_iter": 3000,
        "class_weight": "balanced",
        "multi_class": "auto",
        "random_state": seed,
    }

    # Decision-only search. Training stays identical to exp_028; only final argmax is tuned.
    class_weight_grid = np.round(np.arange(0.60, 1.81, 0.10), 4)
    class_weight_fine_span = 0.15
    class_weight_fine_step = 0.025
    prior_alpha_grid = np.round(np.arange(-0.50, 1.51, 0.10), 4)
    prior_alpha_fine_span = 0.15
    prior_alpha_fine_step = 0.025
    decision_search_top_k = 10

    if use_gpu:
        cat_params["devices"] = "0"


def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(CFG.seed)
print(COMP_CONFIG)

## 2. Data Loading

In [ ]:
def find_data_dir(candidates):
    for data_dir in candidates:
        if (data_dir / "train.csv").exists():
            return data_dir
    raise FileNotFoundError("Could not find train.csv. Check Kaggle input settings.")


CFG.data_dir = find_data_dir(CFG.data_dir_candidates)

train = pd.read_csv(CFG.data_dir / "train.csv")
test = pd.read_csv(CFG.data_dir / "test.csv")
sample_submission = pd.read_csv(CFG.data_dir / "sample_submission.csv")

id_col = "id" if "id" in train.columns else sample_submission.columns[0]
if CFG.target and CFG.target in train.columns:
    target_col = CFG.target
else:
    target_col = [c for c in sample_submission.columns if c != id_col][0]

print("data_dir:", CFG.data_dir)
print("train:", train.shape)
print("test :", test.shape)
print("target:", target_col)
print(train.head())

## 3. Feature Generation Hooks

In [ ]:
def add_feature_engineering(df):
    # Episode-specific features go here.
    # Keep exp_001 raw-feature only until an experiment explicitly adds features.
    return df.copy()


def add_basic_features(df):
    # Future extraction point: src/features.py
    return add_feature_engineering(df)


def prepare_features(train_df, test_df):
    train_df = add_basic_features(train_df)
    test_df = add_basic_features(test_df)

    features = [c for c in train_df.columns if c not in [id_col, target_col]]
    all_df = pd.concat([train_df[features], test_df[features]], axis=0, ignore_index=True)

    cat_cols = [c for c in features if all_df[c].dtype == "object" or str(all_df[c].dtype) == "category"]
    num_cols = [c for c in features if c not in cat_cols]

    for c in num_cols:
        median = all_df[c].median()
        train_df[c] = train_df[c].fillna(median)
        test_df[c] = test_df[c].fillna(median)

    for c in cat_cols:
        train_df[c] = train_df[c].astype("string").fillna("__MISSING__")
        test_df[c] = test_df[c].astype("string").fillna("__MISSING__")

        cats = pd.Index(pd.concat([train_df[c], test_df[c]], axis=0).unique())
        train_df[c] = pd.Categorical(train_df[c], categories=cats)
        test_df[c] = pd.Categorical(test_df[c], categories=cats)

    return train_df, test_df, features, cat_cols


train_fe, test_fe, features, cat_cols = prepare_features(train, test)

print("n_features:", len(features))
print("categorical:", cat_cols)

## 4. Validation

In [ ]:
from sklearn.metrics import balanced_accuracy_score


def encode_target(values):
    classes = pd.Index(pd.unique(values))
    class_to_id = {label: i for i, label in enumerate(classes)}
    encoded = pd.Series(values).map(class_to_id).to_numpy()
    return encoded, classes


def make_stratified_folds(y, n_splits, seed):
    rng = np.random.default_rng(seed)
    fold_indices = [[] for _ in range(n_splits)]

    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        for fold, part in enumerate(np.array_split(cls_idx, n_splits)):
            fold_indices[fold].extend(part.tolist())

    all_idx = np.arange(len(y))
    folds = []
    for valid_idx in fold_indices:
        valid_idx = np.array(sorted(valid_idx))
        train_idx = np.setdiff1d(all_idx, valid_idx)
        folds.append((train_idx, valid_idx))
    return folds


def accuracy(y_true, proba):
    return float((y_true == proba.argmax(axis=1)).mean())


def balanced_accuracy(y_true, proba):
    return float(balanced_accuracy_score(y_true, proba.argmax(axis=1)))


def multiclass_log_loss(y_true, proba):
    proba = np.clip(proba, 1e-15, 1 - 1e-15)
    return float(-np.mean(np.log(proba[np.arange(len(y_true)), y_true])))


def class_prior(y_true, n_classes):
    counts = np.bincount(y_true, minlength=n_classes).astype(np.float64)
    return counts / counts.sum()


def adjusted_scores(proba, class_weights=None, prior_alpha=0.0, priors=None):
    scores = np.asarray(proba, dtype=np.float64).copy()
    if priors is not None and prior_alpha != 0:
        scores = scores / np.power(np.asarray(priors, dtype=np.float64), prior_alpha)
    if class_weights is not None:
        scores = scores * np.asarray(class_weights, dtype=np.float64)
    return scores


def predict_with_adjustment(proba, class_weights=None, prior_alpha=0.0, priors=None):
    return adjusted_scores(proba, class_weights, prior_alpha, priors).argmax(axis=1)


def weighted_balanced_accuracy(y_true, proba, class_weights, prior_alpha=0.0, priors=None):
    pred = predict_with_adjustment(proba, class_weights, prior_alpha, priors)
    return float(balanced_accuracy_score(y_true, pred))


def _weight_candidates_around(center, span, step, anchor_idx):
    values = []
    for i, w in enumerate(center):
        if i == anchor_idx:
            values.append(np.array([1.0], dtype=np.float64))
        else:
            lo = max(0.05, float(w) - span)
            hi = float(w) + span + step / 2
            values.append(np.round(np.arange(lo, hi, step), 4))
    return values


def _evaluate_decision_grid(y_true, proba, priors, weight_values, alpha_values, classes, anchor_idx):
    records = []
    for alpha in alpha_values:
        base_scores = adjusted_scores(proba, prior_alpha=float(alpha), priors=priors)
        for w0 in weight_values[0]:
            for w1 in weight_values[1]:
                for w2 in weight_values[2]:
                    weights = np.array([w0, w1, w2], dtype=np.float64)
                    pred = (base_scores * weights).argmax(axis=1)
                    score = float(balanced_accuracy_score(y_true, pred))
                    pred_rate = np.bincount(pred, minlength=len(classes)) / len(pred)
                    records.append({
                        "score": score,
                        "prior_alpha": float(alpha),
                        "class_weights": weights,
                        "pred_rate": pred_rate,
                        "anchor_class": str(classes[anchor_idx]),
                    })
    return records


def tune_decision_adjustment(y_true, proba, priors, classes, anchor_idx=0):
    baseline_weights = np.ones(proba.shape[1], dtype=np.float64)
    baseline_score = weighted_balanced_accuracy(y_true, proba, baseline_weights, 0.0, priors)

    coarse_values = []
    for i in range(proba.shape[1]):
        if i == anchor_idx:
            coarse_values.append(np.array([1.0], dtype=np.float64))
        else:
            coarse_values.append(np.asarray(CFG.class_weight_grid, dtype=np.float64))

    records = _evaluate_decision_grid(
        y_true, proba, priors, coarse_values, CFG.prior_alpha_grid, classes, anchor_idx
    )
    records.append({
        "score": baseline_score,
        "prior_alpha": 0.0,
        "class_weights": baseline_weights,
        "pred_rate": np.bincount(proba.argmax(axis=1), minlength=len(classes)) / len(proba),
        "anchor_class": str(classes[anchor_idx]),
    })

    coarse_top = sorted(records, key=lambda r: r["score"], reverse=True)[:CFG.decision_search_top_k]
    fine_records = []
    for rec in coarse_top:
        fine_values = _weight_candidates_around(
            rec["class_weights"], CFG.class_weight_fine_span, CFG.class_weight_fine_step, anchor_idx
        )
        alpha_lo = float(rec["prior_alpha"]) - CFG.prior_alpha_fine_span
        alpha_hi = float(rec["prior_alpha"]) + CFG.prior_alpha_fine_span + CFG.prior_alpha_fine_step / 2
        fine_alphas = np.round(np.arange(alpha_lo, alpha_hi, CFG.prior_alpha_fine_step), 4)
        fine_records.extend(
            _evaluate_decision_grid(y_true, proba, priors, fine_values, fine_alphas, classes, anchor_idx)
        )

    all_records = records + fine_records
    best = max(all_records, key=lambda r: r["score"])
    rows = []
    for rec in sorted(all_records, key=lambda r: r["score"], reverse=True)[:50]:
        row = {
            "score": rec["score"],
            "prior_alpha": rec["prior_alpha"],
            "anchor_class": rec["anchor_class"],
        }
        for label, weight, rate in zip(classes, rec["class_weights"], rec["pred_rate"]):
            row[f"weight_{label}"] = float(weight)
            row[f"pred_rate_{label}"] = float(rate)
        rows.append(row)

    search_df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    return best["class_weights"], float(best["prior_alpha"]), float(best["score"]), search_df


y, classes = encode_target(train_fe[target_col])
n_classes = len(classes)
folds = make_stratified_folds(y, CFG.n_splits, CFG.seed)
class_priors = class_prior(y, n_classes)

print("classes:", list(classes))
print("folds:", len(folds))
print("train priors:", dict(zip(classes, class_priors)))

## 5. Training

In [ ]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


def encode_categoricals(df):
    out = df.copy()
    for c in cat_cols:
        out[c] = out[c].cat.codes.astype("int32")
    return out


def scale_for_nn(X_train, X_valid, X_test):
    X_train_nn = encode_categoricals(X_train).astype("float32")
    X_valid_nn = encode_categoricals(X_valid).astype("float32")
    X_test_nn = encode_categoricals(X_test).astype("float32")

    scaler = StandardScaler()
    X_train_nn = scaler.fit_transform(X_train_nn)
    X_valid_nn = scaler.transform(X_valid_nn)
    X_test_nn = scaler.transform(X_test_nn)
    return X_train_nn, X_valid_nn, X_test_nn


def train_lgbm_fold(X_train, y_train, X_valid, y_valid, X_test):
    # Future extraction point: src/train.py
    params = CFG.lgbm_params.copy()
    params["num_class"] = n_classes

    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
    valid_data = lgb.Dataset(X_valid, label=y_valid, categorical_feature=cat_cols, free_raw_data=False)

    model = lgb.train(
        params,
        train_data,
        num_boost_round=CFG.num_boost_round,
        valid_sets=[valid_data],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(CFG.early_stopping_rounds, verbose=False),
            lgb.log_evaluation(CFG.verbose_eval),
        ],
    )
    valid_pred = model.predict(X_valid, num_iteration=model.best_iteration)
    test_pred = model.predict(X_test, num_iteration=model.best_iteration)
    return model, valid_pred, test_pred, model.best_iteration


def train_xgb_fold(X_train, y_train, X_valid, y_valid, X_test):
    # Future extraction point: src/train.py
    params = CFG.xgb_params.copy()
    params["num_class"] = n_classes

    X_train_xgb = encode_categoricals(X_train)
    X_valid_xgb = encode_categoricals(X_valid)
    X_test_xgb = encode_categoricals(X_test)

    dtrain = xgb.DMatrix(X_train_xgb, label=y_train)
    dvalid = xgb.DMatrix(X_valid_xgb, label=y_valid)
    dtest = xgb.DMatrix(X_test_xgb)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=CFG.num_boost_round,
        evals=[(dvalid, "valid")],
        early_stopping_rounds=CFG.early_stopping_rounds,
        verbose_eval=CFG.verbose_eval,
    )
    valid_pred = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
    test_pred = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
    return model, valid_pred, test_pred, model.best_iteration


def to_catboost_frame(df):
    out = df.copy()
    for c in cat_cols:
        out[c] = out[c].astype(str)
    return out


def train_cat_fold(X_train, y_train, X_valid, y_valid, X_test):
    # Future extraction point: src/train.py
    cat_features = [X_train.columns.get_loc(c) for c in cat_cols]

    X_train_cat = to_catboost_frame(X_train)
    X_valid_cat = to_catboost_frame(X_valid)
    X_test_cat = to_catboost_frame(X_test)

    train_pool = Pool(X_train_cat, y_train, cat_features=cat_features)
    valid_pool = Pool(X_valid_cat, y_valid, cat_features=cat_features)
    test_pool = Pool(X_test_cat, cat_features=cat_features)

    model = CatBoostClassifier(**CFG.cat_params)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=CFG.early_stopping_rounds,
        use_best_model=True,
    )
    valid_pred = model.predict_proba(valid_pool)
    test_pred = model.predict_proba(test_pool)
    return model, valid_pred, test_pred, model.get_best_iteration()


def train_mlp_fold(X_train, y_train, X_valid, y_valid, X_test):
    # Lightweight NN-style tabular base model. Kept simple to avoid disturbing the notebook structure.
    X_train_nn, X_valid_nn, X_test_nn = scale_for_nn(X_train, X_valid, X_test)
    model = MLPClassifier(**CFG.mlp_params)
    model.fit(X_train_nn, y_train)
    valid_pred = model.predict_proba(X_valid_nn)
    test_pred = model.predict_proba(X_test_nn)
    return model, valid_pred, test_pred, getattr(model, "n_iter_", None)


def train_model(model_name):
    oof = np.zeros((len(train_fe), n_classes), dtype=np.float32)
    test_pred = np.zeros((len(test_fe), n_classes), dtype=np.float32)
    fold_logs = []

    print(f"\n===== {model_name} =====")
    for fold, (tr_idx, va_idx) in enumerate(folds, 1):
        X_tr = train_fe.iloc[tr_idx][features]
        y_tr = y[tr_idx]
        X_va = train_fe.iloc[va_idx][features]
        y_va = y[va_idx]
        X_te = test_fe[features]

        if model_name == "lgbm":
            model, valid_pred, test_fold_pred, best_iteration = train_lgbm_fold(X_tr, y_tr, X_va, y_va, X_te)
        elif model_name == "xgb":
            model, valid_pred, test_fold_pred, best_iteration = train_xgb_fold(X_tr, y_tr, X_va, y_va, X_te)
        elif model_name == "cat":
            model, valid_pred, test_fold_pred, best_iteration = train_cat_fold(X_tr, y_tr, X_va, y_va, X_te)
        elif model_name == "mlp":
            model, valid_pred, test_fold_pred, best_iteration = train_mlp_fold(X_tr, y_tr, X_va, y_va, X_te)
        else:
            raise ValueError(f"Unknown model: {model_name}")

        oof[va_idx] = valid_pred
        test_pred += test_fold_pred / CFG.n_splits

        fold_acc = accuracy(y_va, valid_pred)
        fold_bacc = balanced_accuracy(y_va, valid_pred)
        fold_ll = multiclass_log_loss(y_va, valid_pred)
        fold_logs.append({
            "fold": fold,
            "accuracy": fold_acc,
            "balanced_accuracy": fold_bacc,
            "logloss": fold_ll,
            "best_iteration": int(best_iteration) if best_iteration is not None else None,
        })
        print(
            f"{model_name} fold {fold}: accuracy={fold_acc:.6f}, "
            f"balanced_accuracy={fold_bacc:.6f}, logloss={fold_ll:.6f}, "
            f"best_iteration={best_iteration}"
        )

        del model, X_tr, X_va, X_te
        gc.collect()

    cv_acc = accuracy(y, oof)
    cv_bacc = balanced_accuracy(y, oof)
    cv_ll = multiclass_log_loss(y, oof)
    print(f"{model_name} CV: accuracy={cv_acc:.6f}, balanced_accuracy={cv_bacc:.6f}, logloss={cv_ll:.6f}")

    return {
        "name": model_name,
        "oof": oof,
        "test_pred": test_pred,
        "cv_accuracy": cv_acc,
        "cv_balanced_accuracy": cv_bacc,
        "cv_logloss": cv_ll,
        "folds": fold_logs,
    }


model_results = []
for model_name in CFG.models:
    model_results.append(train_model(model_name))

## 5.5 Logistic Regression Stacking

In [ ]:
from sklearn.linear_model import LogisticRegression


def blend_predictions(results, model_weights=None):
    if model_weights is None:
        blend_weights = np.ones(len(results), dtype=np.float32) / len(results)
    else:
        blend_weights = np.array(model_weights, dtype=np.float32)
        blend_weights = blend_weights / blend_weights.sum()

    blended_oof = sum(w * result["oof"] for w, result in zip(blend_weights, results))
    blended_test = sum(w * result["test_pred"] for w, result in zip(blend_weights, results))
    return blended_oof, blended_test, blend_weights


def make_stack_matrix(results):
    names = [result["name"] for result in results]
    oof_matrix = np.hstack([result["oof"] for result in results])
    test_matrix = np.hstack([result["test_pred"] for result in results])
    return names, oof_matrix, test_matrix


def fit_logreg_stack(results):
    names, oof_matrix, test_matrix = make_stack_matrix(results)
    meta_oof = np.zeros((len(y), n_classes), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(folds, 1):
        meta = LogisticRegression(**CFG.logreg_stack_params)
        meta.fit(oof_matrix[tr_idx], y[tr_idx])
        meta_oof[va_idx] = meta.predict_proba(oof_matrix[va_idx])

        fold_acc = accuracy(y[va_idx], meta_oof[va_idx])
        fold_bacc = balanced_accuracy(y[va_idx], meta_oof[va_idx])
        fold_ll = multiclass_log_loss(y[va_idx], meta_oof[va_idx])
        print(
            f"logreg_stack fold {fold}: accuracy={fold_acc:.6f}, "
            f"balanced_accuracy={fold_bacc:.6f}, logloss={fold_ll:.6f}"
        )

    final_meta = LogisticRegression(**CFG.logreg_stack_params)
    final_meta.fit(oof_matrix, y)
    meta_test = final_meta.predict_proba(test_matrix)

    cv_acc = accuracy(y, meta_oof)
    cv_bacc = balanced_accuracy(y, meta_oof)
    cv_ll = multiclass_log_loss(y, meta_oof)
    class_weights, prior_alpha, tuned_bacc, decision_search_df = tune_decision_adjustment(
        y, meta_oof, class_priors, classes
    )

    print(
        f"logreg_stack CV: accuracy={cv_acc:.6f}, balanced_accuracy={cv_bacc:.6f}, "
        f"logloss={cv_ll:.6f}, tuned_balanced_accuracy={tuned_bacc:.6f}, "
        f"prior_alpha={prior_alpha:.4f}"
    )
    print("stack base models:", names)
    print("best class weights:", dict(zip(classes, class_weights)))
    display(decision_search_df.head(20))

    return {
        "name": "logreg_stack",
        "base_model_names": names,
        "oof": meta_oof,
        "test_pred": meta_test,
        "cv_accuracy": cv_acc,
        "cv_balanced_accuracy": cv_bacc,
        "cv_logloss": cv_ll,
        "class_weights": class_weights,
        "tuned_balanced_accuracy": tuned_bacc,
        "meta_model": final_meta,
    }


stack_result = fit_logreg_stack(model_results) if CFG.use_logreg_stack else None

# Legacy path retained for compatibility. Disabled by default in this experiment.
def make_oof_feature_frames(oof, test_pred, class_labels, prefix):
    train_meta = pd.DataFrame(index=train_fe.index)
    test_meta = pd.DataFrame(index=test_fe.index)

    feature_cols = []
    for i, label in enumerate(class_labels):
        safe_label = str(label).lower().replace(" ", "_")
        col = f"{prefix}_proba_{safe_label}"
        train_meta[col] = oof[:, i]
        test_meta[col] = test_pred[:, i]
        feature_cols.append(col)

    train_sorted = np.sort(oof, axis=1)
    test_sorted = np.sort(test_pred, axis=1)

    for col, train_values, test_values in [
        (f"{prefix}_pred_conf", oof.max(axis=1), test_pred.max(axis=1)),
        (f"{prefix}_margin_top1_top2", train_sorted[:, -1] - train_sorted[:, -2], test_sorted[:, -1] - test_sorted[:, -2]),
    ]:
        train_meta[col] = train_values
        test_meta[col] = test_values
        feature_cols.append(col)

    return train_meta, test_meta, feature_cols


if CFG.use_oof_features:
    print("Adding OOF features and retraining second-stage model")

    stage1_model_results = model_results
    stage1_model_names = [result["name"] for result in stage1_model_results]
    stage1_oof_proba, stage1_test_proba, stage1_weights = blend_predictions(
        stage1_model_results,
        CFG.model_weights,
    )

    existing_oof_cols = [
        c for c in train_fe.columns
        if c.startswith(f"{CFG.oof_feature_prefix}_")
    ]
    if existing_oof_cols:
        train_fe = train_fe.drop(columns=existing_oof_cols)
        test_fe = test_fe.drop(columns=[c for c in existing_oof_cols if c in test_fe.columns])
        features = [c for c in features if c not in existing_oof_cols]

    oof_train_features, oof_test_features, oof_feature_cols = make_oof_feature_frames(
        stage1_oof_proba,
        stage1_test_proba,
        classes,
        CFG.oof_feature_prefix,
    )

    train_fe = pd.concat([train_fe, oof_train_features], axis=1)
    test_fe = pd.concat([test_fe, oof_test_features], axis=1)
    features = features + oof_feature_cols

    model_results = []
    for model_name in CFG.models:
        model_results.append(train_model(model_name))

    print("stage1 models:", stage1_model_names)
    print("added OOF features:", oof_feature_cols)
    print("n_features:", len(features))
else:
    oof_feature_cols = []

## 6. Inference

In [ ]:
if stack_result is not None:
    model_names = [stack_result["name"]]
    weights = np.ones(1, dtype=np.float32)
    oof_proba = stack_result["oof"]
    test_proba = stack_result["test_pred"]

    # Manual decision override for submission
    prior_alpha = 1.20
    class_weights = np.array([1.0, 1.65, 0.775], dtype=np.float64)

    # If this cell is run after an old in-memory stack_result, compute the new
    # decision-only search here instead of failing on missing keys.
    if "prior_alpha" in stack_result and "decision_search_df" in stack_result:
        class_weights = stack_result["class_weights"]
        prior_alpha = stack_result["prior_alpha"]
        decision_search_df = stack_result["decision_search_df"]
    else:
        print("stack_result has no prior_alpha; recomputing decision adjustment from existing OOF probabilities.")
        class_weights, prior_alpha, _, decision_search_df = tune_decision_adjustment(
            y, oof_proba, class_priors, classes
        )
        stack_result["class_weights"] = class_weights
        stack_result["prior_alpha"] = prior_alpha
        stack_result["decision_search_df"] = decision_search_df
else:
    if CFG.model_weights is None:
        weights = np.ones(len(model_results), dtype=np.float32) / len(model_results)
    else:
        weights = np.array(CFG.model_weights, dtype=np.float32)
        weights = weights / weights.sum()

    model_names = [result["name"] for result in model_results]
    oof_proba = sum(w * result["oof"] for w, result in zip(weights, model_results))
    test_proba = sum(w * result["test_pred"] for w, result in zip(weights, model_results))
    class_weights, prior_alpha, _, decision_search_df = tune_decision_adjustment(
        y, oof_proba, class_priors, classes
    )

cv_acc = accuracy(y, oof_proba)
cv_bacc = balanced_accuracy(y, oof_proba)
cv_ll = multiclass_log_loss(y, oof_proba)
tuned_cv_bacc = weighted_balanced_accuracy(
    y, oof_proba, class_weights, prior_alpha=prior_alpha, priors=class_priors
)

use_weighted_decision = tuned_cv_bacc > cv_bacc
if use_weighted_decision:
    test_pred_idx = predict_with_adjustment(
        test_proba, class_weights, prior_alpha=prior_alpha, priors=class_priors
    )
else:
    test_pred_idx = test_proba.argmax(axis=1)

test_pred = classes.to_numpy()[test_pred_idx]
pred_distribution = pd.Series(test_pred).value_counts(normalize=True).sort_index()

oof_adjusted_pred = classes.to_numpy()[
    predict_with_adjustment(oof_proba, class_weights, prior_alpha=prior_alpha, priors=class_priors)
]
oof_adjusted_distribution = pd.Series(oof_adjusted_pred).value_counts(normalize=True).sort_index()

print("models:", model_names)
print("weights:", dict(zip(model_names, weights)))
print(f"ensemble CV: accuracy={cv_acc:.6f}, balanced_accuracy={cv_bacc:.6f}, logloss={cv_ll:.6f}")
print(
    f"tuned balanced_accuracy={tuned_cv_bacc:.6f}, "
    f"use_weighted_decision={use_weighted_decision}, prior_alpha={prior_alpha:.4f}"
)
print("class_weights:", dict(zip(classes, class_weights)))
print("OOF adjusted distribution:")
print(oof_adjusted_distribution)
print("test prediction distribution:")
print(pred_distribution)
display(decision_search_df.head(20))

## 7. Submission

In [ ]:
def upload_submission_to_github(file_path, exp_id):
    import base64
    import requests
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    token = secrets.get_secret("GITHUB_TOKEN")
    owner = secrets.get_secret("GITHUB_OWNER")
    repo = secrets.get_secret("GITHUB_REPO")

    github_path = f"submissions/{exp_id}.csv"
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{github_path}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
    }

    content = base64.b64encode(Path(file_path).read_bytes()).decode()
    payload = {
        "message": f"upload {exp_id} submission",
        "content": content,
    }

    current = requests.get(url, headers=headers, timeout=60)
    if current.status_code == 200:
        payload["sha"] = current.json()["sha"]
    elif current.status_code != 404:
        raise RuntimeError(f"GitHub file check failed: {current.status_code} {current.text[:300]}")

    response = requests.put(url, headers=headers, json=payload, timeout=60)
    if response.status_code not in [200, 201]:
        raise RuntimeError(f"GitHub upload failed: {response.status_code} {response.text[:300]}")

    return {
        "uploaded": True,
        "github_path": github_path,
        "status_code": response.status_code,
    }

In [ ]:
def save_submission(submission, exp_id):
    CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

    archive_path = CFG.artifact_dir / f"{exp_id}.csv"
    submission.to_csv(CFG.submission_path, index=False)
    submission.to_csv(archive_path, index=False)

    info = {
        "submitted": CFG.submitted,
        "public_lb": CFG.public_lb,
        "kaggle_path": str(CFG.submission_path),
        "archive_path": str(archive_path),
        "github_upload": None,
    }

    if CFG.upload_submission_to_github:
        info["github_upload"] = upload_submission_to_github(archive_path, exp_id)

    return info


submission = sample_submission.copy()
submission[target_col] = test_pred

submission_info = save_submission(submission, CFG.exp_id)

print(submission.head())
print("saved:", submission_info)

## 8. Inserting Experiment Log

In [ ]:
def build_experiment_log():
    created_at = datetime.now(timezone.utc).isoformat()

    return {
        "competition": CFG.competition,
        "exp_id": CFG.exp_id,
        "parent_exp_id": CFG.parent_exp_id,
        "note": CFG.note,
        "created_at": created_at,
        "submission": submission_info,
        "metrics": {
            "cv_accuracy": float(cv_acc),
            "cv_balanced_accuracy": float(cv_bacc),
            "cv_logloss": float(cv_ll),
            "tuned_cv_balanced_accuracy": float(tuned_cv_bacc),
            "use_weighted_decision": bool(use_weighted_decision),
            "class_weights": {str(k): float(v) for k, v in zip(classes, class_weights)},
            "prior_alpha": float(prior_alpha),
            "decision_search_top": decision_search_df.head(20).to_dict(orient="records"),
            "oof_adjusted_distribution": {str(k): float(v) for k, v in oof_adjusted_distribution.items()},
            "test_pred_distribution": {str(k): float(v) for k, v in pred_distribution.items()},
        },
        "data": {
            "n_train": int(len(train)),
            "n_test": int(len(test)),
            "target": target_col,
            "classes": list(classes),
        },
        "features": {
            "n_features": len(features),
            "columns": list(features),
            "categorical_columns": list(cat_cols),
            "oof_feature_columns": list(globals().get("oof_feature_cols", [])),
        },
        "models": {
            "names": model_names,
            "weights": dict(zip(model_names, [float(w) for w in weights])),
            "stack": None if stack_result is None else {
                "name": stack_result["name"],
                "base_model_names": stack_result["base_model_names"],
                "cv_accuracy": float(stack_result["cv_accuracy"]),
                "cv_balanced_accuracy": float(stack_result["cv_balanced_accuracy"]),
                "cv_logloss": float(stack_result["cv_logloss"]),
                "tuned_balanced_accuracy": float(stack_result["tuned_balanced_accuracy"]),
                "prior_alpha": float(stack_result["prior_alpha"]),
            },
            "scores": [
                {
                    "name": result["name"],
                    "cv_accuracy": float(result["cv_accuracy"]),
                    "cv_balanced_accuracy": float(result.get("cv_balanced_accuracy", np.nan)),
                    "cv_logloss": float(result["cv_logloss"]),
                    "folds": result["folds"],
                }
                for result in model_results
            ],
        },
        "params": {
            "seed": CFG.seed,
            "n_splits": CFG.n_splits,
            "use_gpu": CFG.use_gpu,
            "num_boost_round": CFG.num_boost_round,
            "early_stopping_rounds": CFG.early_stopping_rounds,
            "lgbm": CFG.lgbm_params,
            "xgb": CFG.xgb_params,
            "cat": CFG.cat_params,
            "mlp": CFG.mlp_params,
            "logreg_stack": CFG.logreg_stack_params,
            "decision_search": {
                "class_weight_grid": list(map(float, CFG.class_weight_grid)),
                "class_weight_fine_span": float(CFG.class_weight_fine_span),
                "class_weight_fine_step": float(CFG.class_weight_fine_step),
                "prior_alpha_grid": list(map(float, CFG.prior_alpha_grid)),
                "prior_alpha_fine_span": float(CFG.prior_alpha_fine_span),
                "prior_alpha_fine_step": float(CFG.prior_alpha_fine_step),
                "top_k": int(CFG.decision_search_top_k),
            },
        },
    }


experiment_log = build_experiment_log()
experiment_log

In [ ]:
def save_experiment_to_neon(experiment_log):
    import json
    import psycopg2
    from kaggle_secrets import UserSecretsClient

    database_url = UserSecretsClient().get_secret("NEON_DATABASE_URL")

    with psycopg2.connect(database_url) as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO experiments (
                    exp_id,
                    parent_exp_id,
                    competition,
                    created_at,
                    note,
                    ensemble_cv_accuracy,
                    ensemble_cv_logloss,
                    n_features,
                    submitted,
                    public_lb,
                    features,
                    categorical_features,
                    oof_feature_columns,
                    model_names,
                    model_weights,
                    model_scores,
                    params,
                    gpu_settings
                )
                VALUES (
                    %s, %s, %s, %s, %s,
                    %s, %s, %s, %s, %s,
                    %s::jsonb, %s::jsonb, %s::jsonb, %s::jsonb, %s::jsonb,
                    %s::jsonb, %s::jsonb, %s::jsonb
                )
                ON CONFLICT (exp_id) DO UPDATE SET
                    parent_exp_id = EXCLUDED.parent_exp_id,
                    competition = EXCLUDED.competition,
                    created_at = EXCLUDED.created_at,
                    note = EXCLUDED.note,
                    ensemble_cv_accuracy = EXCLUDED.ensemble_cv_accuracy,
                    ensemble_cv_logloss = EXCLUDED.ensemble_cv_logloss,
                    n_features = EXCLUDED.n_features,
                    submitted = EXCLUDED.submitted,
                    public_lb = EXCLUDED.public_lb,
                    features = EXCLUDED.features,
                    categorical_features = EXCLUDED.categorical_features,
                    oof_feature_columns = EXCLUDED.oof_feature_columns,
                    model_names = EXCLUDED.model_names,
                    model_weights = EXCLUDED.model_weights,
                    model_scores = EXCLUDED.model_scores,
                    params = EXCLUDED.params,
                    gpu_settings = EXCLUDED.gpu_settings
                """,
                (
                    experiment_log["exp_id"],
                    experiment_log["parent_exp_id"],
                    experiment_log["competition"],
                    experiment_log["created_at"],
                    experiment_log["note"],
                    experiment_log["metrics"]["tuned_cv_balanced_accuracy"],
                    experiment_log["metrics"]["cv_logloss"],
                    experiment_log["features"]["n_features"],
                    experiment_log["submission"]["submitted"],
                    experiment_log["submission"]["public_lb"],
                    json.dumps(experiment_log["features"]["columns"]),
                    json.dumps(experiment_log["features"]["categorical_columns"]),
                    json.dumps(experiment_log["features"].get("oof_feature_columns", [])),
                    json.dumps(experiment_log["models"]["names"]),
                    json.dumps(experiment_log["models"]["weights"]),
                    json.dumps(experiment_log["models"]["scores"]),
                    json.dumps(experiment_log["params"]),
                    json.dumps({
                        "use_gpu": experiment_log["params"]["use_gpu"],
                        "lgbm_device": experiment_log["params"]["lgbm"].get("device"),
                        "xgb_device": experiment_log["params"]["xgb"].get("device"),
                        "cat_task_type": experiment_log["params"]["cat"].get("task_type"),
                    }),
                ),
            )

    return {
        "saved_to_db": True,
        "exp_id": experiment_log["exp_id"],
        "oof_feature_columns": experiment_log["features"].get("oof_feature_columns", []),
    }

In [ ]:
if CFG.save_experiment_to_db:
    db_result = save_experiment_to_neon(experiment_log)
else:
    db_result = {"saved_to_db": False}

db_result